# Experiment 21: Numerical Identity Target Encoding

The previous experiments treated numerical columns primarily as continuous variables. This experiment tests whether the exact numeric values themselves carry categorical-like signal.\n
This is motivated by public experimentation on the same competition, where exact-value target encoding of numerical columns produced the largest reported improvement.\n
All target encodings are leakage-safe: training rows receive out-of-fold encodings, while validation rows use mappings learned only from the training split.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
DATA_PATH = PROJECT_ROOT / 'data' / 'train.csv'
        
train = pd.read_csv(DATA_PATH)

X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

print('Train shape:', X_train.shape)
print('Validation shape:', X_valid.shape)
print('Numeric columns:', numeric_features)
print('Categorical columns:', categorical_features)
print('Numeric column count:', len(numeric_features))


In [ ]:
def clean_values(df, columns):
    result = df.copy()
    for col in columns:
        result[col] = result[col].astype('string').fillna('__MISSING__')
    return result


def build_mapping(values, target, smoothing):
    temp = pd.DataFrame({
        'value': values.values,
        'target': target.values
    })

    global_mean = float(target.mean())
    stats = temp.groupby('value')['target'].agg(['mean', 'count'])

    smoothed = (
        stats['count'] * stats['mean'] + smoothing * global_mean
    ) / (stats['count'] + smoothing)

    return smoothed, global_mean


def add_oof_numeric_target_encoding(X_tr, y_tr, X_va, columns, smoothing):
    train_out = X_tr.copy()
    valid_out = X_va.copy()

    splitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for col in columns:
        train_values = train_out[col].astype('string').fillna('__MISSING__')
        valid_values = valid_out[col].astype('string').fillna('__MISSING__')

        oof = pd.Series(index=train_out.index, dtype='float64')

        for fit_positions, holdout_positions in splitter.split(train_out, y_tr):
            fit_index = train_out.index[fit_positions]
            holdout_index = train_out.index[holdout_positions]

            mapping, global_mean = build_mapping(
                train_values.loc[fit_index],
                y_tr.loc[fit_index],
                smoothing
            )

            oof.loc[holdout_index] = (
                train_values.loc[holdout_index]
                .map(mapping)
                .fillna(global_mean)
                .astype(float)
            )

        full_mapping, full_global_mean = build_mapping(
            train_values,
            y_tr,
            smoothing
        )

        validation_encoded = (
            valid_values
            .map(full_mapping)
            .fillna(full_global_mean)
            .astype(float)
        )

        train_out[f'{col}__identity_target'] = oof
        valid_out[f'{col}__identity_target'] = validation_encoded

    return train_out, valid_out


def add_numeric_frequency_encoding(X_tr, X_va, columns):
    train_out = X_tr.copy()
    valid_out = X_va.copy()

    for col in columns:
        train_values = train_out[col].astype('string').fillna('__MISSING__')
        valid_values = valid_out[col].astype('string').fillna('__MISSING__')

        frequencies = train_values.value_counts(normalize=True)

        train_out[f'{col}__identity_freq'] = (
            train_values.map(frequencies).fillna(0.0).astype(float)
        )

        valid_out[f'{col}__identity_freq'] = (
            valid_values.map(frequencies).fillna(0.0).astype(float)
        )

    return train_out, valid_out


In [ ]:
def prepare_variant(X_tr, y_tr, X_va, variant, smoothing):
    train_data = X_tr.copy()
    valid_data = X_va.copy()

    if variant in ['frequency', 'frequency_target']:
        train_data, valid_data = add_numeric_frequency_encoding(
            train_data,
            valid_data,
            numeric_features
        )

    if variant in ['target', 'frequency_target', 'target_original', 'target_original_onehot']:
        train_data, valid_data = add_oof_numeric_target_encoding(
            train_data,
            y_tr,
            valid_data,
            numeric_features,
            smoothing
        )

    return train_data, valid_data


def build_model():
    return XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )


def fit_score(X_tr, y_tr, X_va, y_va, keep_original_categories):
    train_data = X_tr.copy()
    valid_data = X_va.copy()
        
    if not keep_original_categories:
        train_data = train_data.drop(columns=categorical_features)
        valid_data = valid_data.drop(columns=categorical_features)

    numeric_cols = train_data.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = train_data.select_dtypes(exclude=['number']).columns.tolist()

    transformers = []

    if numeric_cols:
        transformers.append(
            (
                'num',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='median'))
                ]),
                numeric_cols
            )
        )

    if categorical_cols:
        transformers.append(
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('onehot', OneHotEncoder(handle_unknown='ignore'))
                ]),
                categorical_cols
            )
        )

    preprocessor = ColumnTransformer(transformers)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', build_model())
    ])

    pipeline.fit(train_data, y_tr)
    predictions = pipeline.predict_proba(valid_data)[:, 1]
        
    return roc_auc_score(y_va, predictions)


In [ ]:
results = []

configs = [
    ('21A_Frequency', 'frequency', 0, False),
    ('21B_Target_Smoothing_5', 'target', 5, False),
    ('21C_Target_Smoothing_20', 'target', 20, False),
    ('21D_Target_Smoothing_50', 'target', 50, False),
    ('21E_Frequency_Target_20', 'frequency_target', 20, False),
    ('21F_Frequency_Target_50', 'frequency_target', 50, False),
    ('21G_Target_Original_OneHot', 'target_original_onehot', 20, True)
]

for name, variant, smoothing, keep_original_categories in configs:
    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)

    X_tr_variant, X_va_variant = prepare_variant(
        X_train,
        y_train,
        X_valid,
        variant,
        smoothing
    )

    encoded_columns = [
        c for c in X_tr_variant.columns
        if '__identity_target' in c or '__identity_freq' in c
    ]

    print('Original feature count:', X_train.shape[1])
    print('Final feature count:', X_tr_variant.shape[1])
    print('Identity encoded columns:', len(encoded_columns))

    score = fit_score(
        X_tr_variant,
        y_train,
        X_va_variant,
        y_valid,
        keep_original_categories
    )

    results.append({
        'Experiment': name,
        'Features': X_tr_variant.shape[1],
        'Encoded_Columns': len(encoded_columns),
        'ROC_AUC': score
    })

    print(f'ROC-AUC: {score:.6f}')


In [ ]:
results_df = pd.DataFrame(results).sort_values(
    'ROC_AUC',
    ascending=False
).reset_index(drop=True)

print('\n' + '=' * 70)
print('EXPERIMENT 21 RESULTS')
print('=' * 70)
print(results_df.to_string(index=False))

best_score = float(results_df.loc[0, 'ROC_AUC'])
best_name = results_df.loc[0, 'Experiment']
previous_best = 0.941815
difference = best_score - previous_best

print('\nPrevious local best:', f'{previous_best:.6f}')
print('Best Experiment 21 model:', best_name)
print('Best Experiment 21 ROC-AUC:', f'{best_score:.6f}')
print('Difference vs previous best:', f'{difference:+.6f}')

if best_score > previous_best:
    print('\n🔥🔥🔥 NEW LOCAL BEST 🔥🔥🔥')
else:
    print('\nNo Experiment 21 model beat the current local best.')
